[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m2_tool_registry.ipynb) [![Course](https://img.shields.io/badge/Full_Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-2)

# Module 2 — Tool Design & the Agent-Computer Interface

**Level:** Medium | **Time:** ~1.5h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-2)

### What you'll build
A `ToolRegistry` with schema validation, idempotency classification, and automatic error-to-observation wrapping.

### Key concepts
- **JSON schema as documentation**: verbose descriptions beat terse ones — the description is the model's only API doc
- **Tool cardinality**: empirical accuracy drops past 20–40 tools; use hierarchical catalogs
- **Idempotency**: mark every tool — idempotent tools are safe to retry; non-idempotent need confirmation gates
- **Error wrapping**: always return errors as string observations, never raise exceptions into the loop
- **Enum constraints**: use `"enum": ["apa", "mla"]` to make invalid inputs structurally impossible (poka-yoke)

### Research refs
- CodeAct — Wang et al. (2024) https://arxiv.org/abs/2402.01030
- Toolformer — Schick et al. (2023) https://arxiv.org/abs/2302.01318

---

In [ ]:
!pip install openai --quiet

In [ ]:
import json
import math
from typing import Any, Callable

class ToolRegistry:
    """
    Registry with schema validation, idempotency classification,
    and automatic error-to-observation wrapping.
    """

    def __init__(self):
        self._tools: dict[str, dict] = {}       # name -> {fn, schema, idempotent}

    def register(self, fn: Callable, schema: dict, idempotent: bool = True):
        name = schema["name"]
        self._tools[name] = {"fn": fn, "schema": schema, "idempotent": idempotent}

    @property
    def schemas(self) -> list[dict]:
        """Return OpenAI-compatible tool schemas for all registered tools."""
        return [
            {"type": "function", "function": t["schema"]}
            for t in self._tools.values()
        ]

    def execute(self, name: str, args: dict) -> str:
        """Execute a tool, wrapping all errors as string observations."""
        if name not in self._tools:
            return f"ERROR: tool '{name}' not found. Available: {list(self._tools.keys())}"
        entry = self._tools[name]
        try:
            result = entry["fn"](**args)
            return str(result)
        except TypeError as e:
            return f"ERROR: bad arguments for '{name}': {e}"
        except Exception as e:
            return f"ERROR: {name} failed — {type(e).__name__}: {e}"

    def is_idempotent(self, name: str) -> bool:
        return self._tools.get(name, {}).get("idempotent", False)


# ── Tool implementations ──────────────────────────────────────────────────────

def calculate(expression: str) -> float:
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    return eval(expression, {"__builtins__": {}}, allowed)

def store_finding(source: str, passage: str, relevance_score: float) -> str:
    # In production: write to episodic memory / vector store
    return f"Stored: [{relevance_score:.2f}] {source[:40]}..."

def generate_citation(title: str, year: int, author: str = "", url: str = "", style: str = "apa") -> str:
    if style == "apa":
        base = f"{author + '. ' if author else ''}({year}). {title}."
        return f"{base} {url}" if url else base
    if style == "mla":
        return f"{author + '. ' if author else ''}"{title}." {year}."
    return f"{title} ({year})"


# ── Register tools ────────────────────────────────────────────────────────────

registry = ToolRegistry()

registry.register(
    fn=calculate,
    schema={
        "name": "calculator",
        "description": "Evaluate a mathematical expression. Examples: '20.1 / 12.3', 'sqrt(144)'.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Valid math expression."}
            },
            "required": ["expression"]
        }
    },
    idempotent=True
)

registry.register(
    fn=store_finding,
    schema={
        "name": "store_finding",
        "description": "Store a research finding with source, passage, and relevance score.",
        "parameters": {
            "type": "object",
            "properties": {
                "source": {"type": "string"},
                "passage": {"type": "string"},
                "relevance_score": {"type": "number", "description": "0.0–1.0"}
            },
            "required": ["source", "passage", "relevance_score"]
        }
    },
    idempotent=False  # side effect: writes to store
)

registry.register(
    fn=generate_citation,
    schema={
        "name": "generate_citation",
        "description": "Format a citation. Style: apa | mla | chicago.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "year": {"type": "integer"},
                "author": {"type": "string"},
                "url": {"type": "string"},
                "style": {"type": "string", "enum": ["apa", "mla", "chicago"], "default": "apa"}
            },
            "required": ["title", "year"]
        }
    },
    idempotent=True
)


# ── Smoke tests ───────────────────────────────────────────────────────────────

print("Schemas:", json.dumps(registry.schemas[0], indent=2)[:300], "...\n")

print("calculator(20.1 / 12.3):", registry.execute("calculator", {"expression": "20.1 / 12.3"}))
print("calculator(bad expr):", registry.execute("calculator", {"expression": "import os"}))
print("unknown_tool:", registry.execute("nonexistent", {}))

print("store_finding idempotent?", registry.is_idempotent("store_finding"))
print("calculator idempotent?", registry.is_idempotent("calculator"))

print("citation:", registry.execute("generate_citation", {"title": "ReAct", "year": 2022, "author": "Yao et al.", "style": "apa"}))

## Exercise

Design 4 tool schemas for a document research agent.

> **Interview question:** *"How do you design a tool schema to minimize model confusion? Walk me through a bad schema and how you'd fix it."*

In [ ]:
# Exercise: Design tool schemas for a document research agent
# For each tool: write the JSON schema, classify idempotency, justify.

tools_to_design = [
    "search_web(query, num_results)",
    "extract_pdf(url, max_chars)",
    "store_finding(source, passage, relevance_score)",
    "generate_answer(findings: list)"
]

# TODO: implement each as a ToolRegistry.register() call
# - Include at least one usage example in the description field
# - Classify idempotent=True/False and explain why
# - Add enum constraints where appropriate (e.g. extract format)

# Bonus: what happens if you call store_finding twice with the same passage?
# Add deduplication logic to the store_finding implementation.